# LangChain Fundamentals
### Setting up models, sending messages, and understanding LangChain's core primitives

---
**Topics Covered:**
- Installing dependencies
- Loading API keys from `.env`
- Initializing LLMs via `init_chat_model`
- Message types: `SystemMessage`, `HumanMessage`, `AIMessage`
- `invoke()` with different prompts
- Inspecting `AIMessage` response metadata


## 1. Install Required Packages

In [ ]:
# Run once to install dependencies
# !pip install langchain langchain-groq langchain-google-genai python-dotenv

## 2. Load Environment Variables

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY    = os.getenv("GROQ_API_KEY")
GOOGLE_API_KEY  = os.getenv("GOOGLE_API_KEY")
OPENAI_API_KEY  = os.getenv("OPENAI_API_KEY")

# Quick sanity check — prints True if key was loaded
print("Groq key loaded  :", bool(GROQ_API_KEY))
print("Google key loaded:", bool(GOOGLE_API_KEY))

## 3. Initialize a Chat Model with `init_chat_model`

`init_chat_model` is a provider-agnostic factory that returns a `BaseChatModel`.  
Swap the model name/provider string without touching the rest of your code.

In [ ]:
from langchain.chat_models import init_chat_model

# Using an open-source model served on Groq's inference API
llm = init_chat_model(
    "llama-3.3-70b-versatile",
    model_provider="groq",
    temperature=0.7,
)

print(type(llm))

## 4. Understanding LangChain Message Types

| Class | Role | When to use |
|-------|------|-------------|
| `SystemMessage` | system | Sets the assistant's persona / constraints |
| `HumanMessage` | user | The user's question or instruction |
| `AIMessage` | assistant | The model's previous reply (in multi-turn) |

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# Build a conversation manually
conversation = [
    SystemMessage(content="You are a concise Python tutor. Keep answers under 80 words."),
    HumanMessage(content="What is a Python decorator and why would I use one?"),
]

response = llm.invoke(conversation)
print(response.content)

## 5. Inspecting the `AIMessage` Object

The returned object carries more than just `.content` — token counts, finish reason, model name, and latency are all accessible.

In [ ]:
print("Type          :", type(response))
print("Finish reason :", response.response_metadata.get("finish_reason"))
print("Model used    :", response.response_metadata.get("model_name"))
print("Input tokens  :", response.usage_metadata.get("input_tokens"))
print("Output tokens :", response.usage_metadata.get("output_tokens"))

## 6. Multi-Turn Conversation

Append the model's last reply as an `AIMessage` to maintain conversational context across turns.

In [ ]:
# Turn 1
history = [
    SystemMessage(content="You are a concise Python tutor."),
    HumanMessage(content="Explain list comprehension in one sentence."),
]
turn1 = llm.invoke(history)
print("Assistant (turn 1):", turn1.content)

# Turn 2 — append previous reply and ask a follow-up
history.append(turn1)   # AIMessage
history.append(HumanMessage(content="Show me a quick example with squares of even numbers."))

turn2 = llm.invoke(history)
print("\nAssistant (turn 2):", turn2.content)

## 7. Switching Providers — Google Gemini

The same `invoke()` interface works across providers. Change `model_provider` and that's it.

In [ ]:
gemini = init_chat_model(
    "google_genai:gemini-2.0-flash",
    temperature=0.3,
)

answer = gemini.invoke("Name three lesser-known Python standard-library modules and what each does.")
print(answer.content)

## 8. Plain String Shortcut

Passing a raw string to `invoke()` is the same as wrapping it in a single `HumanMessage` — handy for quick experiments.

In [ ]:
quick = llm.invoke("In three bullet points, explain how garbage collection works in CPython.")
print(quick.content)

---
### Summary

| Concept | Key class / method |
|---------|--------------------|
| Model factory | `init_chat_model(model, model_provider)` |
| Send messages | `llm.invoke([...messages...])` |
| Conversation context | Append `AIMessage` between turns |
| Token / metadata | `response.usage_metadata`, `response.response_metadata` |

**Next → `02_prompt_templates.ipynb`**